# Chapter 2 - Recursive State Estimation: Discrete Bayes Filter

The two operations of the Bayes filter:
- Prediction: push uncertainty through the motion model (convolution with motion kernel)
- Correction: keep states that explain the measurement (reweighting the belief)

World: 1D hallway, 10m long, 3 identical doors at x=2.0, 5.0, 8.0m
Belief: discrete grid of 100 cells, each cell 0.1m wide

In [1]:
import sys
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_filters')
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_experiments')
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pluto_filters.bayes_filter.discrete_bayes_filter import (
    DiscreteBayesFilter1D, CELL_CENTERS, DOOR_POSITIONS, N_CELLS
)

def entropy(b):
    b = np.clip(b, 1e-300, 1)
    return float(-np.sum(b * np.log(b)))

print("Setup complete")
print(f"N_CELLS={N_CELLS}, DOOR_POSITIONS={DOOR_POSITIONS}")

Setup complete
N_CELLS=100, DOOR_POSITIONS=[2.0, 5.0, 8.0]


In [2]:
bf = DiscreteBayesFilter1D()
history, events = [], []

def record(ev):
    history.append(bf.belief.copy())
    events.append(ev)

record('prior (uniform)')
bf.update(True);  record('sense door -> 3 peaks')
bf.predict(3.0);  record('move +3m -> peaks shift')
bf.update(True);  record('sense door -> 2 peaks')
bf.predict(3.0);  record('move +3m')
bf.update(True);  record('sense door -> 1 peak (localized)')

fig, axes = plt.subplots(len(history), 1, figsize=(12, 2.5 * len(history)))
fig.suptitle("Discrete Bayes Filter - Pluto's Belief Over Time", fontsize=13)
for i, (belief, ev) in enumerate(zip(history, events)):
    ax = axes[i]
    ax.bar(CELL_CENTERS, belief, width=0.09, color='steelblue', alpha=0.8)
    for d in DOOR_POSITIONS:
        ax.axvline(d, color='brown', ls='--', alpha=0.6)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, belief.max()*1.3+1e-5)
    ax.set_ylabel('p(x)')
    ax.set_title(f'Step {i}: {ev}  [H={entropy(belief):.3f} nats]')
    if i == len(history)-1:
        ax.set_xlabel('Position [m]')
plt.tight_layout()
plt.show()
print(f"Final mode: x = {CELL_CENTERS[np.argmax(bf.belief)]:.2f} m")

Final mode: x = 8.05 m


## Exercise 1 - Negative Evidence: Sense NO Door

When the sensor reports no door, probability drops near all door positions.
Negative evidence is often more informative than positive evidence because
it eliminates multiple hypotheses at once.

In [3]:
bf_neg = DiscreteBayesFilter1D()
neg_history, neg_events = [], []

def rneg(ev):
    neg_history.append(bf_neg.belief.copy())
    neg_events.append(ev)

rneg('prior (uniform)')
bf_neg.update(False); rneg('sense NO door -> anti-peaks at door positions')
bf_neg.predict(1.5);  rneg('move +1.5m')
bf_neg.update(True);  rneg('sense door (now more informative)')
bf_neg.predict(3.0);  rneg('move +3m')
bf_neg.update(True);  rneg('sense door -> localized')

fig, axes = plt.subplots(len(neg_history), 1, figsize=(12, 2.3 * len(neg_history)))
fig.suptitle('Negative Evidence: Sense NO door first, then sense door', fontsize=12)
for i, (belief, ev) in enumerate(zip(neg_history, neg_events)):
    ax = axes[i]
    ax.bar(CELL_CENTERS, belief, width=0.09, color='darkorange', alpha=0.8)
    for d in DOOR_POSITIONS:
        ax.axvline(d, color='brown', ls='--', alpha=0.5)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, belief.max()*1.3+1e-5)
    ax.set_ylabel('p(x)')
    ax.set_title(f'Step {i}: {ev}  [H={entropy(belief):.3f}]')
    if i == len(neg_history)-1:
        ax.set_xlabel('Position [m]')
plt.tight_layout()
plt.show()
print("After sensing NO door, probability drops near all 3 door positions.")
print("The filter uses absence of evidence as evidence of absence.")

After sensing NO door, probability drops near all 3 door positions.
The filter uses absence of evidence as evidence of absence.


## Exercise 2 - Prediction as Convolution

The prediction step is a convolution of the current belief with the motion kernel.
The kernel shape (its width) determines how much uncertainty is added per step.
A wider kernel = more uncertain motion = faster entropy growth.

In [4]:
from scipy.ndimage import gaussian_filter1d

sigma_m = 0.5   # motion uncertainty in meters
sigma_cells = sigma_m / 0.1  # convert to cells
shift_cells = int(3.0 / 0.1)  # 3m shift

belief_before = history[1].copy()
belief_convolved = gaussian_filter1d(belief_before, sigma=sigma_cells, mode='wrap')
belief_after = np.roll(belief_convolved, shift_cells)
belief_after /= belief_after.sum()

kernel_x = np.linspace(-2, 2, 200)
kernel_y = np.exp(-kernel_x**2 / (2 * sigma_m**2))
kernel_y /= kernel_y.sum() * (kernel_x[1] - kernel_x[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Prediction = Convolution with Motion Kernel', fontsize=13)

axes[0].bar(CELL_CENTERS, belief_before, width=0.09, color='steelblue', alpha=0.8)
axes[0].set_title('Before predict: p(x)')
axes[0].set_xlim(0, 10); axes[0].set_xlabel('x [m]'); axes[0].set_ylabel('p(x)')

axes[1].fill_between(kernel_x, kernel_y, alpha=0.7, color='green')
axes[1].set_title(f'Motion kernel (sigma={sigma_m}m, shift=3m)')
axes[1].set_xlabel('displacement [m]')

axes[2].bar(CELL_CENTERS, belief_after, width=0.09, color='coral', alpha=0.8)
axes[2].set_title('After predict: peaks spread and shift')
axes[2].set_xlim(0, 10); axes[2].set_xlabel('x [m]')

plt.tight_layout()
plt.show()
print(f"Entropy before predict: {entropy(belief_before):.3f}")
print(f"Entropy after predict : {entropy(belief_after):.3f}")
print("Prediction increases entropy (spreads uncertainty).")

Entropy before predict: 4.542
Entropy after predict : 4.600
Prediction increases entropy (spreads uncertainty).


## Exercise 3 - Wrong Map: Confident but Wrong

When the map has incorrect door positions, the filter converges with high confidence
to the wrong answer. The filter cannot detect that its map is wrong.
This is why map quality matters: incorrect priors produce confidently incorrect posteriors.

In [5]:
class WrongMapFilter(DiscreteBayesFilter1D):
    WRONG_DOORS = [3.0, 6.0, 9.0]

    def update(self, door_detected):
        likelihood = np.ones(N_CELLS) * 0.15
        for d in self.WRONG_DOORS:
            idx = int(d / 0.1)
            r = 5
            likelihood[max(0, idx-r):min(N_CELLS, idx+r)] = 0.85
        if not door_detected:
            likelihood = 1.0 - likelihood
        self.belief *= likelihood
        s = self.belief.sum()
        if s > 0:
            self.belief /= s

wrong_bf = WrongMapFilter()
wrong_bf.update(True)
wrong_bf.predict(3.0)
wrong_bf.update(True)
wrong_bf.predict(3.0)
wrong_bf.update(True)

correct_final = history[-1]
wrong_final   = wrong_bf.belief.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Wrong Map: Confident in the Wrong Answer', fontsize=13)

ax = axes[0]
ax.bar(CELL_CENTERS, correct_final, width=0.09, color='steelblue', alpha=0.8)
for d in DOOR_POSITIONS:
    ax.axvline(d, color='brown', ls='--', alpha=0.6, label='true door')
ax.set_title(f'Correct map - peak at x={CELL_CENTERS[np.argmax(correct_final)]:.1f}m')
ax.set_xlim(0, 10); ax.legend(fontsize=8, loc='upper left')
ax.set_xlabel('x [m]'); ax.set_ylabel('p(x)')

ax = axes[1]
ax.bar(CELL_CENTERS, wrong_final, width=0.09, color='crimson', alpha=0.8)
for d in [3.0, 6.0, 9.0]:
    ax.axvline(d, color='green', ls='--', alpha=0.6, label='wrong door pos')
ax.set_title(f'Wrong map - peak at x={CELL_CENTERS[np.argmax(wrong_final)]:.1f}m  <- WRONG')
ax.set_xlim(0, 10); ax.legend(fontsize=7, loc='upper left')
ax.set_xlabel('x [m]')

plt.tight_layout()
plt.show()
print(f"Correct map final entropy: {entropy(correct_final):.4f}")
print(f"Wrong   map final entropy: {entropy(wrong_final):.4f}")
print("Both have similar entropy. The filter has no way to detect map errors.")

Correct map final entropy: 4.0203
Wrong   map final entropy: 3.3972
Both have similar entropy. The filter has no way to detect map errors.


## Exercise 4 - Sensor Reliability Sweep

Sweeping the hit probability from near-perfect (0.95) to random (0.50).
At $p_{\text{hit}} = 0.50$ the sensor is no better than a coin flip, and the filter
cannot localize regardless of how many observations are made.

In [6]:
p_hits = [0.95, 0.75, 0.60, 0.50]
colors = ['green', 'steelblue', 'orange', 'crimson']

fig, axes = plt.subplots(1, len(p_hits), figsize=(15, 4))
fig.suptitle('Sensor Reliability Sweep - After 3 door observations', fontsize=13)

for ax, p_hit, col in zip(axes, p_hits, colors):
    class SweepFilter(DiscreteBayesFilter1D):
        def __init__(self, p):
            super().__init__()
            self._p = p
        def update(self, door_detected):
            likelihood = np.ones(N_CELLS) * (1 - self._p)
            for d in DOOR_POSITIONS:
                idx = int(d / 0.1)
                r = 5
                likelihood[max(0, idx-r):min(N_CELLS, idx+r)] = self._p
            if not door_detected:
                likelihood = 1.0 - likelihood
            self.belief *= likelihood
            s = self.belief.sum()
            if s > 0:
                self.belief /= s

    sf = SweepFilter(p_hit)
    sf.update(True); sf.predict(3.0)
    sf.update(True); sf.predict(3.0)
    sf.update(True)
    H = entropy(sf.belief)
    ax.bar(CELL_CENTERS, sf.belief, width=0.09, color=col, alpha=0.8)
    for d in DOOR_POSITIONS:
        ax.axvline(d, color='brown', ls='--', alpha=0.5)
    ax.set_xlim(0, 10)
    ax.set_title(f'p_hit={p_hit}\nH={H:.3f}')
    ax.set_xlabel('x [m]')
    ax.set_ylabel('p(x)' if ax == axes[0] else '')

plt.tight_layout()
plt.show()
print("p_hit=0.50 means the sensor is random - belief stays nearly uniform.")
print("Sensor quality directly controls how fast the filter localizes.")

p_hit=0.50 means the sensor is random - belief stays nearly uniform.
Sensor quality directly controls how fast the filter localizes.


## Exercise 5 - Entropy Over Time: When Is Localization Good Enough?

Entropy measures how spread out the belief is. High entropy = uncertain.
Low entropy = confident. A threshold on entropy can be used to decide
when the robot knows enough about its position to begin acting.

In [7]:
bf5 = DiscreteBayesFilter1D()
ent_log = [entropy(bf5.belief)]
step_labels = ['start']

bf5.update(True);   ent_log.append(entropy(bf5.belief)); step_labels.append('sense door')
bf5.predict(0.5);   ent_log.append(entropy(bf5.belief)); step_labels.append('move 0.5m')
bf5.predict(0.5);   ent_log.append(entropy(bf5.belief)); step_labels.append('move 0.5m')
bf5.predict(0.5);   ent_log.append(entropy(bf5.belief)); step_labels.append('move 0.5m')
bf5.predict(0.5);   ent_log.append(entropy(bf5.belief)); step_labels.append('move 0.5m')
bf5.predict(0.5);   ent_log.append(entropy(bf5.belief)); step_labels.append('move 0.5m')
bf5.update(True);   ent_log.append(entropy(bf5.belief)); step_labels.append('sense door')
bf5.predict(1.0);   ent_log.append(entropy(bf5.belief)); step_labels.append('move 1m')
bf5.update(False);  ent_log.append(entropy(bf5.belief)); step_labels.append('sense NO door')
bf5.predict(1.0);   ent_log.append(entropy(bf5.belief)); step_labels.append('move 1m')
bf5.update(True);   ent_log.append(entropy(bf5.belief)); step_labels.append('sense door')

THRESHOLD = 0.5

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ent_log, 'b-o', lw=2, ms=6)
ax.axhline(THRESHOLD, color='red', ls='--', lw=1.5, label=f'Action threshold ({THRESHOLD} nats)')
ax.axhline(np.log(N_CELLS), color='gray', ls=':', lw=1.5,
           label=f'Max entropy (uniform) = {np.log(N_CELLS):.2f}')
for i, lbl in enumerate(step_labels):
    ax.annotate(lbl, (i, ent_log[i] + 0.04), rotation=45, ha='left', fontsize=7)
ax.set_xlabel('Step index')
ax.set_ylabel('Entropy [nats]')
ax.set_title('Entropy over time - when is localization good enough to act?')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

first_below = next((i for i, h in enumerate(ent_log) if h < THRESHOLD), None)
if first_below is not None:
    print(f"First step below threshold {{THRESHOLD}} nats: step {{first_below}} ({{step_labels[first_below]}})")
else:
    print(f"Entropy never dropped below {{THRESHOLD}} nats in this run")

Entropy never dropped below {THRESHOLD} nats in this run


## Compare, Break, Measure

In [8]:
# Compare: correct map vs wrong map entropy at convergence
print("=== Compare: Correct map vs Wrong map ===")
print(f"Correct map final entropy: {entropy(correct_final):.4f} nats")
print(f"Wrong map   final entropy: {entropy(wrong_final):.4f} nats")
print("Low entropy in both cases. The filter is equally confident in the wrong answer.")

# Break: 5 doors instead of 3 - harder to disambiguate
class FiveDoorFilter(DiscreteBayesFilter1D):
    FIVE_DOORS = [1.5, 3.0, 4.5, 6.5, 8.5]
    def update(self, door_detected):
        likelihood = np.ones(N_CELLS) * 0.15
        for d in self.FIVE_DOORS:
            idx = int(d / 0.1); r = 4
            likelihood[max(0,idx-r):min(N_CELLS,idx+r)] = 0.85
        if not door_detected:
            likelihood = 1.0 - likelihood
        self.belief *= likelihood
        s = self.belief.sum()
        if s > 0:
            self.belief /= s

fd = FiveDoorFilter()
fd.update(True); fd.predict(3.0); fd.update(True); fd.predict(3.0); fd.update(True)
print(f"\n=== Break: 5 doors instead of 3 ===")
print(f"5-door final entropy: {entropy(fd.belief):.4f} nats")
print(f"3-door final entropy: {entropy(correct_final):.4f} nats")
print("More identical features = harder disambiguation = higher entropy at convergence.")

# Measure: steps to reach H < 0.5 for each sensor quality
print("\n=== Measure: steps to reach H < 0.5 nats ===")
for p_hit in [0.95, 0.75, 0.60]:
    class MF(DiscreteBayesFilter1D):
        def update(self, d):
            L = np.ones(N_CELLS) * (1 - p_hit)
            for dp in DOOR_POSITIONS:
                idx = int(dp/0.1); r = 5
                L[max(0,idx-r):min(N_CELLS,idx+r)] = p_hit
            if not d: L = 1 - L
            self.belief *= L
            s = self.belief.sum()
            if s > 0: self.belief /= s
    mf = MF()
    for step in range(1, 60):
        if step % 2 == 1: mf.update(True)
        else: mf.predict(3.0)
        if entropy(mf.belief) < 0.5:
            print(f"  p_hit={p_hit}: localized at step {step}"); break
    else:
        print(f"  p_hit={p_hit}: NOT localized within 60 steps")

=== Compare: Correct map vs Wrong map ===
Correct map final entropy: 4.0203 nats
Wrong map   final entropy: 3.3972 nats
Low entropy in both cases. The filter is equally confident in the wrong answer.

=== Break: 5 doors instead of 3 ===
5-door final entropy: 3.6742 nats
3-door final entropy: 4.0203 nats
More identical features = harder disambiguation = higher entropy at convergence.

=== Measure: steps to reach H < 0.5 nats ===
  p_hit=0.95: NOT localized within 60 steps
  p_hit=0.75: NOT localized within 60 steps
  p_hit=0.6: NOT localized within 60 steps
